In [ ]:
# ==========================================
# 0. ENVIRONMENT SETUP & DEPENDENCIES
# ==========================================
!pip install -q timm albumentations einops

import os
import re
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



In [ ]:

# ==========================================
# 1. DATASET & MULTI-CLASS MAPPING SETUP
# ==========================================
class PlantSegReplicationDataset(Dataset):
    """
    Dataset class handling the 115-class 'in-the-wild' PlantSeg structure 
    or custom leaf datasets while parsing canonical labels automatically[cite: 20].
    """
    def __init__(self, img_dir, mask_dir, max_samples=None, transform=None):
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform

        img_files = {f.stem: f for f in self.img_dir.glob("*.*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]}
        mask_files = {f.stem: f for f in self.mask_dir.glob("*.*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]}
        
        common_stems = sorted(list(set(img_files.keys()) & set(mask_files.keys())))

        if max_samples and len(common_stems) > max_samples:
            np.random.seed(42)
            common_stems = list(np.random.choice(common_stems, max_samples, replace=False))

        self.samples = [(img_files[s], mask_files[s]) for s in common_stems]

        def clean_label(stem):
            label = re.sub(r'_(google|bing|baidu)', '', stem, flags=re.IGNORECASE)
            label = re.sub(r'_([a-zA-Z]+_[a-zA-Z]+)\\s*\\(\\d+\\)', '', label)
            label = re.sub(r'_([a-zA-Z]+)\\s*\\(\\d+\\)', '', label)
            label = re.sub(r'\\s*\\(\\d+\\)', '', label)
            label = re.sub(r'_\\d+$', '', label)
            return label.lower().strip()

        cleaned_labels = [clean_label(s) for s in common_stems]
        unique_labels = sorted(list(set(cleaned_labels)))
        
        self.label_to_idx = {name: i for i, name in enumerate(unique_labels)}
        self.labels = [self.label_to_idx[l] for l in cleaned_labels]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        raw_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        
        # Keep multi-class mask labels intact (0 = background, 1..N = disease IDs)
        mask = raw_mask.astype(np.int64)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"].long()
        else:
            mask = torch.tensor(mask, dtype=torch.long)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, mask, label



In [ ]:
# ==========================================
# 2. ARCHITECTURE: HBAA MODULE & STAR-NET
# ==========================================
class StripPooling(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        h_str = self.pool_h(x)
        w_str = self.pool_w(x)
        out = h_str.expand_as(x) + w_str.expand_as(x)
        return torch.sigmoid(self.conv(out)) * x

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels)
        )
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        max_out = F.adaptive_max_pool2d(x, (1, 1))
        avg_out = F.adaptive_avg_pool2d(x, (1, 1))
        ca = torch.sigmoid(self.mlp(max_out) + self.mlp(avg_out)).unsqueeze(2).unsqueeze(3)
        x_ca = x * ca
        
        max_s = torch.max(x_ca, dim=1, keepdim=True)[0]
        avg_s = torch.mean(x_ca, dim=1, keepdim=True)[0]
        sa = torch.sigmoid(self.spatial_conv(torch.cat([max_s, avg_s], dim=1)))
        return x_ca * sa

class HBAAModule(nn.Module):
    def __init__(self, channels):
        super().__init__()
        ch_sixth = channels // 6
        self.branch_1x1 = nn.Conv2d(channels, ch_sixth, 1)
        self.branch_window = nn.Sequential(
            nn.Conv2d(ch_sixth, ch_sixth, 3, padding=1, groups=ch_sixth),
            nn.GELU()
        )
        self.branch_atrous6 = nn.Conv2d(ch_sixth, ch_sixth, 3, padding=6, dilation=6, groups=ch_sixth)
        self.branch_atrous12 = nn.Conv2d(ch_sixth, ch_sixth, 3, padding=12, dilation=12, groups=ch_sixth)
        self.branch_strip = StripPooling(ch_sixth)
        self.branch_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, ch_sixth, 1)
        )
        
        self.cbam = CBAM(channels)
        self.fusion_conv = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        p1 = self.branch_1x1(x)
        p2 = self.branch_window(p1)
        p3 = self.branch_atrous6(p1)
        p4 = self.branch_atrous12(p1)
        p5 = self.branch_strip(p1)
        p6 = F.interpolate(self.branch_pool(x), size=(h, w), mode='bilinear', align_corners=False)
        
        concat = torch.cat([p1, p2, p3, p4, p5, p6], dim=1)
        out = self.cbam(concat)
        out = self.fusion_conv(out)
        return out + x  # Residual connection[cite: 22]

class STARNet(nn.Module):
    def __init__(self, num_seg_classes=2):
        super().__init__()
        # Swin Transformer backbone (pretrained ImageNet-1K)[cite: 20, 25]
        self.backbone = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, features_only=True)
        self.hbaa = HBAAModule(768) # Stage 4 output channels
        
        self.shallow_conv = nn.Conv2d(96, 96, 1) # Stage 1 output channels
        self.shallow_cbam = CBAM(96)
        
        self.decoder_conv = nn.Sequential(
            nn.Conv2d(768 + 96, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, num_seg_classes, 1)
        )

    def forward(self, x):
        features = self.backbone(x)
        stage1_feat = features[0]  # 1/4 scale details[cite: 21]
        stage4_feat = features[3]  # 1/32 scale deep semantics[cite: 21]
        
        enhanced_deep = self.hbaa(stage4_feat)
        upsampled_deep = F.interpolate(enhanced_deep, size=stage1_feat.shape[2:], mode='bilinear', align_corners=False)
        
        shallow = self.shallow_cbam(self.shallow_conv(stage1_feat))
        fused = torch.cat([upsampled_deep, shallow], dim=1)
        
        out = self.decoder_conv(fused)
        out = F.interpolate(out, size=x.shape[2:], mode='bilinear', align_corners=False)
        return out




In [ ]:
# ==========================================
# 3. LOSS FUNCTION: DPW-LOSS (3-PHASE CURRICULUM)
# ==========================================
class GeneralizedDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets, num_classes=2):
        preds = F.softmax(preds, dim=1)
        targets_one_hot = F.one_hot(targets.clamp(0, num_classes-1), num_classes=num_classes).permute(0, 3, 1, 2).float()
        
        intersection = torch.sum(preds * targets_one_hot, dim=(0, 2, 3))
        union = torch.sum(preds + targets_one_hot, dim=(0, 2, 3))
        
        weights = 1.0 / (torch.sum(targets_one_hot, dim=(0, 2, 3)) ** 2 + self.smooth)
        gdl = 1.0 - 2.0 * torch.sum(weights * intersection) / (torch.sum(weights * union) + self.smooth)
        return gdl

class DPWLoss(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.num_classes = num_classes
        self.ce = nn.CrossEntropyLoss()
        self.gdl = GeneralizedDiceLoss()

    def focal_loss(self, inputs, targets, alpha=0.25, gamma=2.0):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        return (alpha * (1 - pt) ** gamma * ce_loss).mean()

    def forward(self, inputs, targets, global_step):
        # Exact iteration phase thresholds specified in Fan et al. (2026)[cite: 24]
        if global_step < 80000:
            alpha, beta, gamma = 1.0, 0.1, 0.1  # Phase 1: CE dominant[cite: 23, 24]
        elif global_step < 240000:
            alpha, beta, gamma = 0.1, 0.1, 1.0  # Phase 2: Focal dominant[cite: 24]
        else:
            alpha, beta, gamma = 0.1, 1.0, 0.5  # Phase 3: GDL dominant[cite: 24]

        l_ce = self.ce(inputs, targets)
        l_focal = self.focal_loss(inputs, targets)
        l_gdl = self.gdl(inputs, targets, self.num_classes)

        return (alpha * l_ce) + (beta * l_gdl) + (gamma * l_focal)




In [ ]:
# ==========================================
# 4. TRAINING PIPELINE & EXECUTION
# ==========================================
train_transform = A.Compose([
    A.Resize(512, 512),  # Standardized input resolution[cite: 20, 25]
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

def run_replication_training():
    train_img_folder = "/kaggle/input/datasets/weitianqi/plantseg/plantsegv2/images/train/"
    train_mask_folder = "/kaggle/input/datasets/weitianqi/plantseg/plantsegv2/annotations/train/"

    train_dataset = PlantSegReplicationDataset(
        img_dir=train_img_folder, 
        mask_dir=train_mask_folder, 
        max_samples=None, 
        transform=train_transform
    )
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=4,  # Paper batch size setting[cite: 25]
        shuffle=True, 
        num_workers=2, 
        pin_memory=True
    )

    model = STARNet(num_seg_classes=2).to(device)
    criterion = DPWLoss(num_classes=2)
    
    # Optimizer configuration from paper specs[cite: 25]
    optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5, betas=(0.9, 0.999), weight_decay=0.01)
    
    total_iterations = 320000
    scheduler = torch.optim.lr_scheduler.PolynomialLR(optimizer, total_iters=total_iterations, power=0.9)
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    global_step = 0
    epochs = 2  # Adjust epochs based on available Kaggle run quota limits
    
    print("\nInitialized STAR-Net replication training loop...")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, masks, _ in train_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if device.type == 'cuda':
                with torch.amp.autocast('cuda'):
                    seg_out = model(images)
                    loss = criterion(seg_out, masks, global_step)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                seg_out = model(images)
                loss = criterion(seg_out, masks, global_step)
                loss.backward()
                optimizer.step()

            scheduler.step()
            running_loss += loss.item()
            global_step += 1

        print(f"Epoch [{epoch+1}/{epochs}] | Step [{global_step}] | Running Loss: {running_loss / len(train_loader):.4f}")

    os.makedirs("/kaggle/working", exist_ok=True)
    torch.save(model.state_dict(), "/kaggle/working/starnet_replication_model.pth")
    print("\nTraining complete! Weights saved successfully to '/kaggle/working/starnet_replication_model.pth'.")

# Execute the replication pipeline
run_replication_training()